In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df =spark.read.table('fintech360.silver.upi_data')
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. READ SILVER SOURCE TABLE
# ============================================================

source_df = spark.read.table("fintech360.silver.upi_data")

# ============================================================
# 2. CAST COLUMNS FOR VALIDATION
# Preserve original columns in the final output.
# ============================================================

df = (source_df
    .withColumn("_customer_id", F.col("customer_id").cast("long"))
    .withColumn("_transaction_amount", F.col("transaction_amount").cast("double"))
    .withColumn("_transaction_ts", F.col("transaction_ts").cast("long"))
    .withColumn("_created_at", F.col("created_at").cast("long"))
    .withColumn("_updated_at", F.col("updated_at").cast("long"))
    .withColumn("_transaction_status", F.col("transaction_status").cast("int"))
    .withColumn("_is_refund", F.col("is_refund").cast("int"))
    .withColumn("_upi_txn_id", F.trim(F.col("upi_txn_id")))
    .withColumn("_rrn", F.trim(F.col("rrn")))
    .withColumn("_payer_vpa", F.lower(F.trim(F.col("payer_vpa"))))
    .withColumn("_payee_vpa", F.lower(F.trim(F.col("payee_vpa"))))
    .withColumn("_currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("_merchant_category", F.upper(F.trim(F.col("merchant_category"))))
    .withColumn("_upi_app", F.upper(F.trim(F.col("upi_app"))))
    .withColumn("_device_os", F.upper(F.trim(F.col("device_os"))))
    .withColumn("_payer_account_type", F.upper(F.trim(F.col("payer_account_type"))))
)

# ============================================================
# 3. CONVERT UNIX SECONDS TO TIMESTAMPS
# ============================================================

for c in ["transaction_ts", "created_at", "updated_at"]:
    df = df.withColumn(
        f"_dt_{c}",
        F.to_timestamp(F.from_unixtime(F.col(f"_{c}")))
    )

# ============================================================
# 4. CHECK DUPLICATE TRANSACTION IDS AND RRNs
# ============================================================

txn_window = Window.partitionBy("_upi_txn_id")
rrn_window = Window.partitionBy("_rrn")

df = (df
    .withColumn(
        "_txn_id_count",
        F.when(
            F.col("_upi_txn_id").isNotNull() &
            (F.col("_upi_txn_id") != ""),
            F.count("*").over(txn_window)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "_rrn_count",
        F.when(
            F.col("_rrn").isNotNull() &
            (F.col("_rrn") != ""),
            F.count("*").over(rrn_window)
        ).otherwise(F.lit(0))
    )
)

# ============================================================
# 5. DEFINE DATA QUALITY RULES
# ============================================================

allowed_currencies = ["INR"]
allowed_statuses = [0, 1]
allowed_refund_flags = [0, 1]
allowed_account_types = ["SAVINGS", "CURRENT", "WALLET"]
allowed_device_os = ["ANDROID", "IOS", "WINDOWS", "OTHER"]

rules = [
    # Transaction ID
    (
        F.col("upi_txn_id").isNull() |
        (F.trim(F.col("upi_txn_id")) == ""),
        "MISSING_UPI_TXN_ID"
    ),
    (
        F.col("_txn_id_count") > 1,
        "DUPLICATE_UPI_TXN_ID"
    ),

    # Customer ID
    (
        F.col("customer_id").isNull() |
        F.col("_customer_id").isNull() |
        (F.col("_customer_id") <= 0),
        "INVALID_CUSTOMER_ID"
    ),

    # Payer and payee VPA
    (
        F.col("_payer_vpa").isNull() |
        (~F.col("_payer_vpa").rlike(r"^[^@\s]+@[^@\s]+$")),
        "INVALID_PAYER_VPA"
    ),
    (
        F.col("_payee_vpa").isNull() |
        (~F.col("_payee_vpa").rlike(r"^[^@\s]+@[^@\s]+$")),
        "INVALID_PAYEE_VPA"
    ),

    # Transaction amount
    (
        F.col("transaction_amount").isNull() |
        F.col("_transaction_amount").isNull() |
        F.isnan(F.col("_transaction_amount")) |
        (F.abs(F.col("_transaction_amount")) == float("inf")) |
        (F.col("_transaction_amount") <= 0),
        "INVALID_TRANSACTION_AMOUNT"
    ),

    # Currency
    (
        F.col("_currency").isNull() |
        (~F.col("_currency").isin(allowed_currencies)),
        "INVALID_CURRENCY"
    ),

    # Transaction status
    (
        F.col("transaction_status").isNull() |
        F.col("_transaction_status").isNull() |
        (~F.col("_transaction_status").isin(allowed_statuses)),
        "INVALID_TRANSACTION_STATUS"
    ),

    # Refund flag
    (
        F.col("is_refund").isNull() |
        F.col("_is_refund").isNull() |
        (~F.col("_is_refund").isin(allowed_refund_flags)),
        "INVALID_REFUND_FLAG"
    ),

    # Reference number
    (
        F.col("rrn").isNull() |
        (F.trim(F.col("rrn")) == ""),
        "MISSING_RRN"
    ),
    (
        F.col("_rrn_count") > 1,
        "DUPLICATE_RRN"
    ),

    # Merchant category and UPI application
    (
        F.col("merchant_category").isNull() |
        (F.trim(F.col("merchant_category")) == ""),
        "MISSING_MERCHANT_CATEGORY"
    ),
    (
        F.col("upi_app").isNull() |
        (F.trim(F.col("upi_app")) == ""),
        "MISSING_UPI_APP"
    ),

    # Device operating system
    (
        F.col("_device_os").isNull() |
        (~F.col("_device_os").isin(allowed_device_os)),
        "INVALID_DEVICE_OS"
    ),

    # Payer account type
    (
        F.col("_payer_account_type").isNull() |
        (~F.col("_payer_account_type").isin(allowed_account_types)),
        "INVALID_PAYER_ACCOUNT_TYPE"
    ),

    # Transaction timestamp
    (
        F.col("transaction_ts").isNull() |
        F.col("_transaction_ts").isNull() |
        F.col("_dt_transaction_ts").isNull() |
        (F.col("_dt_transaction_ts") > F.current_timestamp()),
        "INVALID_TRANSACTION_TIMESTAMP"
    ),

    # Created and updated timestamps
    (
        F.col("created_at").isNull() |
        F.col("_created_at").isNull() |
        F.col("_dt_created_at").isNull(),
        "INVALID_CREATED_AT"
    ),
    (
        F.col("updated_at").isNull() |
        F.col("_updated_at").isNull() |
        F.col("_dt_updated_at").isNull(),
        "INVALID_UPDATED_AT"
    ),
    (
        F.col("_dt_created_at").isNotNull() &
        F.col("_dt_updated_at").isNotNull() &
        (F.col("_dt_updated_at") < F.col("_dt_created_at")),
        "UPDATED_BEFORE_CREATED"
    )
]

# ============================================================
# 6. CREATE FAILURE REASONS
# All applicable failures are captured for each record.
# ============================================================

df = df.withColumn(
    "dq_failed_reasons",
    F.concat_ws(
        " | ",
        *[
            F.when(condition, F.lit(reason))
            for condition, reason in rules
        ]
    )
)

# ============================================================
# 7. SPLIT PASSED AND QUARANTINED DATA
# ============================================================

helper_columns = [
    "_customer_id", "_transaction_amount", "_transaction_ts",
    "_created_at", "_updated_at", "_transaction_status",
    "_is_refund", "_upi_txn_id", "_rrn", "_payer_vpa",
    "_payee_vpa", "_currency", "_merchant_category",
    "_upi_app", "_device_os", "_payer_account_type",
    "_dt_transaction_ts", "_dt_created_at", "_dt_updated_at",
    "_txn_id_count", "_rrn_count"
]

original_columns = source_df.columns

upi_data_passed = (
    df.filter(F.col("dq_failed_reasons") == "")
      .select(*original_columns)
)

quarantine_data = (
    df.filter(F.col("dq_failed_reasons") != "")
      .drop(*helper_columns)
)

# ============================================================
# 8. DATA QUALITY SUMMARY
# ============================================================

source_count = source_df.count()
passed_count = upi_data_passed.count()
failed_count = quarantine_data.count()

print(f"Source records      : {source_count}")
print(f"Passed records      : {passed_count}")
print(f"Quarantined records : {failed_count}")
print(f"Reconciled records  : {passed_count + failed_count}")

# Inspect quarantined transactions
display(
    quarantine_data.select(
        "upi_txn_id",
        "customer_id",
        "transaction_amount",
        "transaction_status",
        "rrn",
        "is_refund",
        "dq_failed_reasons"
    )
)

# Inspect valid transactions
display(upi_data_passed)


In [0]:

# Create destination schemas
spark.sql("""
CREATE SCHEMA IF NOT EXISTS fintech360.data_quality_check
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS fintech360.quarantine_data
""")

# Save passed UPI transactions
(upi_data_passed.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.data_quality_check.upi_data"))

# Save quarantined transactions, including failure reasons
(quarantine_data.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.quarantine_data.upi_data"))

print("UPI Data Quality Gate completed.")
